# Push/Fold RL — Fase 4 do plano (seção 17: Colab)

Treina um agente PPO no ambiente `PushFoldOpenEnv`/`PushFoldFacingShoveEnv` (`poker_coach/poker_coach/rl/pushfold_env.py`), reaproveitando o motor Nash já existente (`pushfold/nash.py`) como reward, vilão e referência de avaliação.

**Divisão de trabalho (seção 17 do plano):**
- Máquina local: Postgres, parser, feature extraction, dataset generation, API, UI.
- Colab: recebe o parquet do dataset (Fase 1), produz o artefato do modelo (`pushfold_ppo.zip`).

**Honestidade sobre GPU:** esse ambiente específico tem observação de 3 dimensões e 2 ações — uma rede minúscula. Rodar no Colab foi pedido explicitamente (plano AI Pro), mas o Stable-Baselines3 já avisa que PPO com `MlpPolicy` desse tamanho não ganha muito de GPU (overhead de transferência CPU↔GPU supera o ganho). GPU importa de verdade quando o ambiente crescer (pós-flop, seção 20/roadmap V6+) — aqui o notebook serve pra validar o pipeline completo com mais budget de timesteps do que rodar local.

**Cuidado com leakage do benchmark (seção 16 do plano):** reward, vilão e avaliação usam o MESMO motor Nash. Isso mede se a infraestrutura de RL converge pra uma aproximação do equilíbrio chip-EV já conhecido — não é uma alegação de estratégia superior a Nash.

## 1. Instalar dependências

In [ ]:
!pip install -q stable-baselines3 gymnasium pandas pyarrow

## 2. Clonar o repositório (branch `developer`)

Precisa que o branch esteja pushado pro GitHub antes de rodar isso (`git push origin developer` na máquina local).

In [ ]:
!git clone --branch developer --depth 1 https://github.com/felipesembay/poker_coach.git
import sys
sys.path.insert(0, "/content/poker_coach/poker_coach")

## 3. Upload do dataset (parquet da Fase 1)

Gerado localmente via `python -m poker_coach.cli export-rl --out poker_coach_decisions.parquet`. Usado aqui só pra CALIBRAR a distribuição de `(effective_bb, pot_bb)` dos episódios de treino com spots reais — o reward continua vindo do Nash, não do resultado real da mão (ver `export.py`: outcome != qualidade de decisão).

In [ ]:
from google.colab import files
uploaded = files.upload()  # selecione poker_coach_decisions.parquet
dataset_path = next(iter(uploaded))

In [ ]:
import pandas as pd
from poker_coach.rl.pushfold_env import PushFoldOpenEnv, PushFoldFacingShoveEnv, sample_states_from_dataset, evaluate_against_nash

df = pd.read_parquet(dataset_path)
open_states = sample_states_from_dataset(df, "preflop_open")
facing_states = sample_states_from_dataset(df, "preflop_facing_allin") + sample_states_from_dataset(df, "preflop_multiway_facing_allin")
print(f"Spots calibrados: {len(open_states)} de abertura, {len(facing_states)} facing-shove")

## 4. Treinar PPO (mais timesteps do que o smoke-test local)

Local, um smoke-test com 5.000 timesteps já levou a concordância de 50% (baseline aleatório) pra ~88% em segundos. Aqui usamos um budget bem maior pra ver onde converge de verdade.

In [ ]:
from poker_coach.rl.train_pushfold_ppo import train_and_benchmark, format_training_report

report, model = train_and_benchmark(
    total_timesteps=300_000,
    seed=0,
    state_samples=open_states,
    grid_points=25,
)
print(format_training_report(report))

## 5. Avaliação — as 3 métricas (seção 15 do plano)

`train_and_benchmark` já roda A) agreement e B) EV gap contra uma grade held-out (C, generalização — nunca usada como episódio de treino). Célula abaixo separa visualmente clear vs. borderline, e roda o mesmo agente contra uma segunda grade com seed diferente pra checar estabilidade.

In [ ]:
from poker_coach.rl.train_pushfold_ppo import agent_predict_fn

predict = agent_predict_fn(model)
for seed in (1, 2, 3):
    r = evaluate_against_nash(PushFoldOpenEnv, predict, grid_points=25, seed=seed)
    print(f"seed={seed}: agreement={r.agreement_pct}%  mean_ev_gap={r.mean_ev_gap}  "
          f"clear={r.clear_agreement_pct}%(n={r.clear_n})  borderline={r.borderline_agreement_pct}%(n={r.borderline_n})")

## 6. (Opcional) Facing-Shove Env — mesmo processo, outro lado do equilíbrio

In [ ]:
report_facing, model_facing = train_and_benchmark(
    total_timesteps=300_000, seed=0, state_samples=facing_states, grid_points=25,
)
# train_and_benchmark usa PushFoldOpenEnv internamente por padrão — pra treinar
# o facing-shove, troque a chamada por train_ppo(..., ) com PushFoldFacingShoveEnv
# diretamente (mesma assinatura, ver poker_coach/rl/train_pushfold_ppo.py).

## 7. Exportar o artefato treinado

In [ ]:
from poker_coach.rl.train_pushfold_ppo import save_model

save_model(model, "pushfold_ppo.zip")
files.download("pushfold_ppo.zip")

## Lacuna conhecida (não escondida)

Este notebook treina o PPO **do zero** (init padrão do SB3), não a partir
da Personal Policy (Fase 2, Behavioral Cloning) como pedido na seção 14
do plano. Fazer o BC warm-start direito exige transplantar pesos
pré-treinados pra dentro de `model.policy.mlp_extractor`/`action_net`,
ou usar a lib `imitation` (BC + integração nativa com SB3). Ver docstring
de `poker_coach/poker_coach/rl/train_pushfold_ppo.py` pro caminho
concreto de como fechar essa lacuna numa próxima sessão.